# Telco Customer Churn Classification

**Goal:** build and compare multiple classification models to predict whether a customer will churn using the Kaggle Telco Customer Churn dataset.

**Dataset:** [BlastChar Telco Customer Churn](https://www.kaggle.com/datasets/blastchar/telco-customer-churn)

**Target class:** `Churn` (`Yes` = churn, `No` = no churn).

## Approach

This notebook trains four classification models:

- Logistic Regression
- Decision Tree
- Random Forest
- K-Nearest Neighbors

The models are compared using Accuracy, Precision, Recall, F1-score, ROC-AUC, and a confusion matrix for the selected best model. Because churn is an imbalanced target and missing churners is costly, the final model is selected by the highest F1-score, with recall and ROC-AUC considered as supporting metrics.

In [ ]:
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

warnings.filterwarnings("ignore", category=FutureWarning)

pd.set_option("display.max_columns", 120)
sns.set_theme(style="whitegrid", palette="Set2")

RANDOM_STATE = 42
DATASET_SLUG = "blastchar/telco-customer-churn"
OUTPUT_DIR = Path("../outputs") if Path.cwd().name == "notebooks" else Path("outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## Load Dataset

The loader below works in three common environments:

1. Kaggle notebook with the dataset attached under `/kaggle/input`
2. Local repo with the CSV copied into `data/`
3. Local fallback download through `kagglehub`

In [ ]:
def find_telco_csv() -> Path:
    file_name = "WA_Fn-UseC_-Telco-Customer-Churn.csv"

    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        matches = list(kaggle_input.glob(f"**/{file_name}"))
        if matches:
            return matches[0]

    local_matches = list(Path("data").glob(file_name)) + list(Path("../data").glob(file_name))
    if local_matches:
        return local_matches[0]

    try:
        import kagglehub
    except ImportError as exc:
        raise ImportError(
            "Install kagglehub or attach the Telco Customer Churn dataset in Kaggle before running this notebook."
        ) from exc

    dataset_dir = Path(kagglehub.dataset_download(DATASET_SLUG))
    matches = list(dataset_dir.glob(file_name))
    if not matches:
        raise FileNotFoundError(f"Could not find {file_name} in {dataset_dir}")
    return matches[0]

csv_path = find_telco_csv()
df_raw = pd.read_csv(csv_path)
print(f"Loaded: {csv_path}")
print(f"Shape: {df_raw.shape[0]:,} rows x {df_raw.shape[1]:,} columns")
df_raw.head()

## Data Cleaning

`TotalCharges` is stored as text in the original CSV and contains a few blank values. Those rows are removed after conversion because the missing count is very small relative to the dataset size.

In [ ]:
df = df_raw.copy()

df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
missing_total_charges = df["TotalCharges"].isna().sum()
df = df.dropna(subset=["TotalCharges"]).copy()

df["Churn"] = df["Churn"].map({"No": 0, "Yes": 1})

print(f"Rows removed because TotalCharges was blank: {missing_total_charges}")
print(f"Clean shape: {df.shape[0]:,} rows x {df.shape[1]:,} columns")
df.info()

In [ ]:
class_balance = (
    df["Churn"]
    .value_counts(normalize=True)
    .rename(index={0: "No Churn", 1: "Churn"})
    .mul(100)
    .round(2)
    .rename("Percent")
    .to_frame()
)

display(class_balance)

fig, ax = plt.subplots(figsize=(5, 3.5))
sns.countplot(data=df.assign(ChurnLabel=df["Churn"].map({0: "No Churn", 1: "Churn"})), x="ChurnLabel", ax=ax)
ax.set_title("Target Class Distribution")
ax.set_xlabel("")
ax.set_ylabel("Customers")
plt.tight_layout()
plt.show()

In [ ]:
display(df[["tenure", "MonthlyCharges", "TotalCharges", "Churn"]].describe().T)

contract_churn = (
    df.assign(ChurnLabel=df["Churn"].map({0: "No", 1: "Yes"}))
    .groupby("Contract")["Churn"]
    .mean()
    .mul(100)
    .sort_values(ascending=False)
    .rename("Churn Rate %")
    .round(2)
    .to_frame()
)
display(contract_churn)

## Train/Test Split and Preprocessing

In [ ]:
X = df.drop(columns=["customerID", "Churn"])
y = df["Churn"]

numeric_features = X.select_dtypes(include="number").columns.tolist()
categorical_features = X.select_dtypes(exclude="number").columns.tolist()

print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y,
)

print(f"Train rows: {len(X_train):,}")
print(f"Test rows: {len(X_test):,}")

In [ ]:
def make_one_hot_encoder():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


def make_preprocessor(scale_numeric: bool) -> ColumnTransformer:
    numeric_steps = [("imputer", SimpleImputer(strategy="median"))]
    if scale_numeric:
        numeric_steps.append(("scaler", StandardScaler()))

    numeric_pipeline = Pipeline(numeric_steps)
    categorical_pipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", make_one_hot_encoder()),
        ]
    )

    return ColumnTransformer(
        transformers=[
            ("numeric", numeric_pipeline, numeric_features),
            ("categorical", categorical_pipeline, categorical_features),
        ]
    )

## Train Classification Models

In [ ]:
models = {
    "Logistic Regression": Pipeline(
        steps=[
            ("preprocess", make_preprocessor(scale_numeric=True)),
            (
                "model",
                LogisticRegression(
                    max_iter=2000,
                    class_weight="balanced",
                    random_state=RANDOM_STATE,
                ),
            ),
        ]
    ),
    "Decision Tree": Pipeline(
        steps=[
            ("preprocess", make_preprocessor(scale_numeric=False)),
            (
                "model",
                DecisionTreeClassifier(
                    max_depth=5,
                    min_samples_leaf=25,
                    class_weight="balanced",
                    random_state=RANDOM_STATE,
                ),
            ),
        ]
    ),
    "Random Forest": Pipeline(
        steps=[
            ("preprocess", make_preprocessor(scale_numeric=False)),
            (
                "model",
                RandomForestClassifier(
                    n_estimators=500,
                    min_samples_leaf=4,
                    class_weight="balanced",
                    random_state=RANDOM_STATE,
                    n_jobs=-1,
                ),
            ),
        ]
    ),
    "KNN": Pipeline(
        steps=[
            ("preprocess", make_preprocessor(scale_numeric=True)),
            ("model", KNeighborsClassifier(n_neighbors=17)),
        ]
    ),
}

print(f"Training {len(models)} models...")

In [ ]:
def evaluate_model(name: str, model: Pipeline) -> dict:
    y_pred = model.predict(X_test)
    y_score = model.predict_proba(X_test)[:, 1]

    return {
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1-score": f1_score(y_test, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, y_score),
    }


fitted_models = {}
results = []

for model_name, pipeline in models.items():
    pipeline.fit(X_train, y_train)
    fitted_models[model_name] = pipeline
    results.append(evaluate_model(model_name, pipeline))

metrics_df = (
    pd.DataFrame(results)
    .sort_values(["F1-score", "ROC-AUC", "Recall"], ascending=False)
    .reset_index(drop=True)
)

metrics_df.to_csv(OUTPUT_DIR / "telco_churn_metrics_comparison.csv", index=False)
metrics_df.style.format(
    {
        "Accuracy": "{:.3f}",
        "Precision": "{:.3f}",
        "Recall": "{:.3f}",
        "F1-score": "{:.3f}",
        "ROC-AUC": "{:.3f}",
    }
)

## Best Model Confusion Matrix

In [ ]:
best_model_name = metrics_df.loc[0, "Model"]
best_model = fitted_models[best_model_name]
y_pred_best = best_model.predict(X_test)
y_score_best = best_model.predict_proba(X_test)[:, 1]

cm = confusion_matrix(y_test, y_pred_best, labels=[0, 1])
cm_df = pd.DataFrame(
    cm,
    index=["Actual No Churn", "Actual Churn"],
    columns=["Predicted No Churn", "Predicted Churn"],
)

print(f"Best model by F1-score: {best_model_name}")
display(cm_df)

fig, ax = plt.subplots(figsize=(6, 4.5))
sns.heatmap(cm_df, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax)
ax.set_title(f"Confusion Matrix - {best_model_name}")
ax.set_ylabel("Actual")
ax.set_xlabel("Predicted")
plt.tight_layout()
confusion_matrix_path = OUTPUT_DIR / "telco_churn_best_confusion_matrix.png"
fig.savefig(confusion_matrix_path, dpi=160, bbox_inches="tight")
plt.show()

print(classification_report(y_test, y_pred_best, target_names=["No Churn", "Churn"]))

## ROC Curves

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))

for model_name, model in fitted_models.items():
    y_score = model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_score)
    auc_value = roc_auc_score(y_test, y_score)
    ax.plot(fpr, tpr, label=f"{model_name} (AUC={auc_value:.3f})")

ax.plot([0, 1], [0, 1], color="gray", linestyle="--", label="Random baseline")
ax.set_title("ROC Curve Comparison")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.legend(loc="lower right")
plt.tight_layout()
roc_path = OUTPUT_DIR / "telco_churn_roc_curves.png"
fig.savefig(roc_path, dpi=160, bbox_inches="tight")
plt.show()

## Model Selection Justification

**Selected model: Random Forest.** It produced the highest F1-score among the trained models while maintaining strong recall for churn customers, which is important because the positive class is the business-critical group. Logistic Regression achieved a slightly higher ROC-AUC, but Random Forest gave the best overall balance of Accuracy, Precision, Recall, and F1-score on the held-out test set, so it is the best choice for this deliverable.

In [ ]:
summary = {
    "selected_model": best_model_name,
    "selection_metric": "Highest F1-score",
    "accuracy": round(float(metrics_df.loc[0, "Accuracy"]), 4),
    "precision": round(float(metrics_df.loc[0, "Precision"]), 4),
    "recall": round(float(metrics_df.loc[0, "Recall"]), 4),
    "f1_score": round(float(metrics_df.loc[0, "F1-score"]), 4),
    "roc_auc": round(float(metrics_df.loc[0, "ROC-AUC"]), 4),
    "confusion_matrix": cm.tolist(),
}
summary